# To load base model/merged model finetuned by SFTT, and tested its inference result with arithmatic/CoT input questions.

In [1]:
%%capture
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1" # [NEW] Extra 30% context lengths!
!pip install --upgrade -qqq uv
try: import numpy, PIL; get_numpy = f"numpy=={numpy.__version__}"; get_pil = f"pillow=={PIL.__version__}"
except: get_numpy = "numpy"; get_pil = "pillow"
try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
except: is_t4 = False
get_vllm, get_triton = ("vllm==0.8.5", "triton==3.2.0") 
!uv pip install -qqq --upgrade     unsloth {get_vllm} {get_numpy} {get_pil} torchvision bitsandbytes xformers
!uv pip install -qqq {get_triton}
!uv pip install "huggingface_hub>=0.34.0" "datasets>=3.4.1,<4.0.
!uv pip install transformers==4.53.2
!uv pip install torch==2.6.0
!uv pip install --no-deps trl==0.22.2

In [2]:
#!uv pip install unsloth==2025.7.5
!uv pip install vllm==0.8.5.post1
!uv pip install xformers==0.0.29.post3
!uv pip install triton==3.2.0
!uv pip install bitsandbytes

Using Python 3.11.13 environment at: /usr
Resolved 162 packages in 147ms
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠹ Preparing packages... (0/1)
⠹ Preparing packages... (0/1)
⠹ Preparing packages... (0/1)
⠹ Preparing packages... (0/1)
⠹ Preparing packages... (0/1)
⠸ Preparing packages... (0/1)
⠸ Preparing packages... (0/1)
⠸ Preparing packages... (0/1)
⠸ Preparing packages... (0/1)
⠼ Preparing packages... (0/1)
⠼ Preparing packages... (0/

In [3]:
# Install other dependencies
!uv pip install "huggingface_hub>=0.34.0" "datasets>=3.4.1,<4.0.0"
!uv pip install transformers==4.53.2
!uv pip install torch==2.6.0
!uv pip install --no-deps trl==0.22.2
!uv pip install torchvision

Using Python 3.11.13 environment at: /usr
Resolved 45 packages in 80ms
⠙ Preparing packages... (0/3)
⠙ Preparing packages... (0/3)
⠙ Preparing packages... (0/3)
⠙ Preparing packages... (0/3)
⠙ Preparing packages... (0/3)
⠙ Preparing packages... (0/3)
⠙ Preparing packages... (0/3)
⠙ Preparing packages... (0/3)
dill                 ------------------------------ 78.93 KiB/113.53 KiB
⠙ Preparing packages... (0/3)
dill                 ------------------------------ 78.93 KiB/113.53 KiB
⠙ Preparing packages... (0/3)
dill                 ------------------------------ 78.93 KiB/113.53 KiB
⠙ Preparing packages... (0/3)
dill                 ------------------------------ 78.93 KiB/113.53 KiB
fsspec               ------------------------------     0 B/189.08 KiB
⠙ Preparing packages... (0/3)
dill                 ------------------------------ 94.93 KiB/113.53 KiB
fsspec               ------------------------------     0 B/189.08 KiB
⠙ Preparing packages... (0/3)
dill                 -----------

In [4]:
# --- 1. Configuration ---
import os
import pandas as pd
import torch
import re
import io
import sys
import ast
import time
from transformers import AutoTokenizer
import math
import csv
from decimal import Decimal, getcontext
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [5]:
# Set precision for Decimal calculations
getcontext().prec = 50

# To determine which model to be tested
LOAD_BASE_MODEL = False #True
if LOAD_BASE_MODEL:
    base_model_path = "/kaggle/input/qwen-3/transformers/4b-base/1"
else:
    base_model_path = "/kaggle/input/qwen3-4b-sfft-merged/transformers/default/1/Qwen3-4B-SFFT-merged"

TEST_arithmatic = True

In [6]:
if torch.cuda.is_available():
    device = torch.device("cuda:0")
    print(f"Using device: {device}")
else:
    device = torch.device("cpu")
    print("CUDA is not available. Using CPU.")

Using device: cuda:0


In [7]:
from unsloth import FastLanguageModel
import torch
from peft import PeftModel
max_seq_length = 2048 # Can increase for longer reasoning traces
lora_rank = 32 # Larger rank = smarter, but slower
# --- Load model ---

base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = base_model_path,
    max_seq_length = max_seq_length,
    load_in_4bit = False, # False for LoRA 16bit
    torch_dtype = torch.float16, # Force float16 for T4
    #fast_inference = True, # Enable vLLM fast inference
    max_lora_rank = lora_rank,
    gpu_memory_utilization = 0.9, # Reduce if out of memory
)
if LOAD_BASE_MODEL:
    print("Base model loaded successfully with Unsloth.")   
else:
    print("Merged model loaded successfully with Unsloth.")   

model = base_model

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/tmp/ipykernel_19/1006716142.py:1: UserWarning: WARNING: Unsloth should be imported before transformers to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel
2025-11-20 18:37:50.808878: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763663871.199252      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763663871.319093      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


INFO 11-20 18:38:33 [importing.py:53] Triton module has been replaced with a placeholder.
INFO 11-20 18:38:33 [__init__.py:239] Automatically detected platform cuda.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.11.3: Fast Qwen3 patching. Transformers: 4.53.2. vLLM: 0.8.5.post1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Merged model loaded successfully with Unsloth.


In [8]:
# --- 2. SYSTEM PROMPTS (Must match training) ---
code_system_prompt = \
"""You are an expert Python programmer. Your sole task is to write a self-contained Python script to solve the given computational problem.
- Your response MUST begin directly with the code block ```python and end with ```.
- Do NOT provide any text or explanation before or after the code block.
- The script must define a function `solve()` that returns the final numerical answer.
- The script must then call the `solve()` function. The result should be the final expression of the script.
- Do NOT solve the problem yourself or provide any reasoning inside the script, just return the caculation.
- Analyze the problem carefully and choose the appropriate response format, which should contain non-repetitive answers.
- For example, for 'caculate the value of (1+1)', your entire response must be: '```python
def solve():
    return 1+1
print(solve())
```'.
"""

cot_code_system_prompt = \
"""You are a multi-talented expert in mathematics and Python programming. Your task is to solve the given problem by providing both a textual explanation and a Python script.
- First, provide a clear, step-by-step explanation of your reasoning.
- After the explanation, provide a complete, self-contained Python script inside a ```python ... ``` block.
- The script should define a function `solve()` that returns the final numerical answer.
- The script must then call the `solve()` function. The result should be the final expression of the script.
- For example, for 'caculate the value of (1+1)', your script need to be: '```python
def solve():
    return 1+1
print(solve())
```'.
"""

In [9]:
code_system_prompt = ""
cot_code_system_prompt = ""

In [10]:
# --- 2.1 To determine which type of problem it is and then encapsulate it with respective prompt  
def select_prompt(problem: str) -> str:
    """
    Selects the appropriate system prompt based on the problem's content.
    This should roughly match the logic used to categorize data during training.
    """
    problem_lower = problem.lower()

    if "calculate the" in problem_lower:
        print("(System 01: Detected computational problem, using Code-Only prompt.)")
        #problem = problem_lower.replace("calculate","$Calculate#")
        #problem = problem.replace("Calculate","$Calculate#")
        return code_system_prompt, problem
    
    # Keywords that suggest a need for reasoning/proof (CoT)
    proof_keywords = [
        # Original keywords
        'prove that', 'show that', 'demonstrate that', 'explain why',
        'is it true that', 'determine', 'converge', 'convergent',
        'relationship', 'what is the probability',
        # Added keywords from user examples and common math terms
        'equation', 'derivative', 'expansion', 'how many', 'what is',
        'compute the', 'simplify', 'solve for', 'express', 'theorem',
        'proof', 'show', 'derive', 'relates', 'find an', 'find the', 'calculate the'
    ]
    
    # Mathematical symbols/patterns that often appear in theoretical problems
    proof_symbols = [
        # Original symbols
        r'\\sum', r'\\int', r'\\lim', r'\\infty', r'\\binom', r'\\choose',
        r'\\prod', r'\\partial', r'\\nabla', r'\\subset', r'\\in',
        # Added symbols and patterns
        r'\\theta', r'\\sec', r'\\pi', r'\\alpha', r'\\beta', r'\\gamma',
        r'\\delta', r'\\sin', r'\\cos', r'\\tan', r'\\log',
        # Match 'ln' as a word, or '\\ln' for latex
        r'\\ln', r'\bln\b',
        r'\\sqrt',
        # Match f(x), g(x), etc.
        r'f\(x\)', r'g\(x\)', r'h\(x\)',
        r'd/dx', r'\^',
        # Corrected patterns for escaped parentheses for LaTeX e.g. \\( ... \\)
        r'\\\(' , r'\\\)',
        # Pattern for simple parentheses in computational problems e.g. (1+2)
        # This is a broad match, but problems with parens are often not simple arithmetic.
        r'\('
    ]

    if any(keyword in problem_lower for keyword in proof_keywords) or \
       any(re.search(symbol, problem) for symbol in proof_symbols):
        print("(System 02: Detected theoretical problem, using CoT+Code prompt.)")
        return cot_code_system_prompt, problem
    else:
        print("(System 03: Detected computational problem, using Code-Only prompt.)")
        return code_system_prompt, problem

In [11]:
def run_code(code_string: str) -> tuple[str, bool]:
    """
    Executes a string of Python code, capturing its output.
    Intelligently adds a print statement if the last line is an expression.
    """
    output_buffer = io.StringIO()
    original_stdout = sys.stdout
    scope = {}
    try:
        sys.stdout = output_buffer
        tree = ast.parse(code_string.strip())
        
        # Check if the last node is an expression that is NOT already a print call
        if tree.body and isinstance(tree.body[-1], ast.Expr):
            last_expr = tree.body[-1].value
            # Check if it's a function call and the function name is 'print'
            is_print_call = (isinstance(last_expr, ast.Call) and 
                             isinstance(last_expr.func, ast.Name) and 
                             last_expr.func.id == 'print')
            
            if not is_print_call:
                # It's an expression but not a print call, so wrap it
                last_expr_node = tree.body.pop()
                print_node = ast.Expr(
                    value=ast.Call(
                        func=ast.Name(id='print', ctx=ast.Load()),
                        args=[last_expr_node.value],
                        keywords=[]
                    )
                )
                tree.body.append(ast.fix_missing_locations(print_node))

        exec(compile(tree, '<string>', 'exec'), scope)
    except Exception as e:
        print(f"Error executing code: {e}", file=sys.stderr)
        return f"Error: {e}", True
    finally:
        sys.stdout = original_stdout
        
    return output_buffer.getvalue().strip(), False

In [12]:
# --- Main Evaluation Logic ---

def evaluate_model(num, input_file, output_file):
    try:
        df = pd.read_csv(input_file)
        questions_df = df.head(num)
        print(f"Loaded {len(questions_df)} questions from arithmetic_problems.csv")
    except FileNotFoundError:
        print("Error: arithmetic_problems.csv not found.")
        sys.exit(1)

    evaluation_results = []

    for index, row in questions_df.iterrows():
        current_question = row['arithmetical question']
        ground_truth_answer = row['answer to the question']

        print(f"\n--- Processing Question {index + 1}/{len(questions_df)} ---")
        print(f"Question: {current_question}")

        system_prompt, question_for_model = select_prompt(current_question)
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question_for_model},
        ]
        inputs = tokenizer.apply_chat_template(
            messages, 
            tokenize=True, 
            add_generation_prompt=True, 
            padding=True, 
            return_tensors="pt"
        ).to(device)

        if (system_prompt==code_system_prompt):
            max_length = 1024#512
        else:
             max_length = 2048
        prompt_token_length = inputs.shape[1]
        start_time = time.time()   
        outputs = model.generate(
            input_ids=inputs, 
            max_new_tokens=max_length, 
            use_cache=True, 
            do_sample=False,#True, 
            #temperature=0.2, 
            #top_p=0.9
            eos_token_id=tokenizer.eos_token_id
        )
        outputs_without_prompt = outputs[:, prompt_token_length: ]
        response_text = tokenizer.batch_decode(outputs_without_prompt, skip_special_tokens=True)[0]
        #print("\nFinish reasoning.")
        end_time = time.time()
        ref_time = end_time - start_time
        print("It cost {} seconds for reference".format(ref_time))
        try:
            assistant_response = response_text.split("<|im_start|>assistant\n")[-1].strip()
        except IndexError:
            assistant_response = response_text

        pattern = r"```+(?:python|py)?\s*\n(.*?)\n```+"
        matches = re.findall(pattern, assistant_response, re.DOTALL)
        python_code = ""
        result = ""
        is_correct = False
        revised_content = assistant_response
        print(revised_content)
        if matches:
            python_code = matches[0].strip()
            result, had_error = run_code(python_code)
            if not had_error and result:
                try:
                    result_decimal = Decimal(result)
                    ground_truth_decimal = Decimal(str(ground_truth_answer))
                    if math.isclose(result_decimal, ground_truth_decimal, rel_tol=1e-3):
                        is_correct = True
                    # Use a clean marker for successful output
                    insert_text = f"```python\n{python_code}\n\n---[Code Output]---\n{result}\n```"    
                except Exception:
                    is_correct = False
            else:
                insert_text = f"```python\n{python_code}\n\n---[Code Execution Error]---\n{result}\n```"

            # Replace the original code block with the annotated version
            original_block_in_markdown = f"```python\n{python_code}\n```"
            revised_content = revised_content.replace(original_block_in_markdown, insert_text, 1)
            #print(revised_content)
        
        print("is_correct: ", is_correct)
        evaluation_results.append({
            "question": current_question,
            "ground_truth": ground_truth_answer,
            "llm_response": assistant_response,
            "extracted_code": python_code,
            "execution_result": result,
            "is_correct": is_correct
        })

    
    # --- Write detailed results to CSV ---
    output_csv_path = output_file
    with open(output_csv_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=evaluation_results[0].keys())
        writer.writeheader()
        writer.writerows(evaluation_results)
    print(f"\nDetailed evaluation results saved to {output_csv_path}")

    # --- Calculate and Print Final Accuracy and Incorrect Samples ---
    correct_predictions = sum(1 for r in evaluation_results if r['is_correct'])
    total_questions = len(evaluation_results)
    accuracy = (correct_predictions / total_questions) * 100 if total_questions > 0 else 0

    print("\n\n--- Evaluation Finished ---")
    print(f"Total Questions: {total_questions}")
    print(f"Correct Predictions: {correct_predictions}")
    print(f"Accuracy: {accuracy:.2f}%")

    print("\n--- Incorrect Samples ---")
    incorrect_samples = [r for r in evaluation_results if not r['is_correct']]
    if not incorrect_samples:
        print("None. All samples were answered correctly!")
    else:
        for i, sample in enumerate(incorrect_samples):
            print(f"\n[{i+1}] Question: {sample['question']}")
            print(f"    Ground Truth: {sample['ground_truth']}")
            print(f"    LLM Result:   {sample['execution_result']}")
            print("-" * 20)

In [13]:
#To set the number for testing, usually within [0, 100]
num_for_test = 20#40
#input_file = "/kaggle/input/test-data3/arithmetic_problems_v3.csv"#"/kaggle/input/test-data/arithmetic_problems.csv"
input_file = "/kaggle/input/random-samples-for-inference/random_samples_for_inference.csv"
if LOAD_BASE_MODEL:
    output_file = "/kaggle/working/evaluation_results_base_cot_v5.csv"
else:
    output_file = "/kaggle/working/evaluation_results_merged_cot_v5.csv"
evaluate_model(num_for_test, input_file, output_file)

Loaded 20 questions from arithmetic_problems.csv

--- Processing Question 1/20 ---
Question: Find the maximum value of a positive constant $a$ such that $\sqrt{x + y} + \sqrt{y} \geq \sqrt{x + ay}$ for all $x \geq 0, y \geq 0$.
(System 02: Detected theoretical problem, using CoT+Code prompt.)
It cost 56.794206380844116 seconds for reference
To find the maximum value of the positive constant \( a \) such that the inequality \(\sqrt{x + y} + \sqrt{y} \geq \sqrt{x + ay}\) holds for all \( x \geq 0 \) and \( y \geq 0 \), we start by analyzing the inequality at specific values of \( x \) and \( y \).

First, let's set \( x = 0 \). The inequality becomes:
\[
\sqrt{0 + y} + \sqrt{y} \geq \sqrt{0 + ay} \implies 2\sqrt{y} \geq \sqrt{ay}.
\]
Squaring both sides, we get:
\[
4y \geq ay \implies 4 \geq a.
\]
This tells us that \( a \leq 4 \). Now, we need to check if \( a = 4 \) satisfies the original inequality for all \( x \geq 0 \) and \( y \geq 0 \).

Substitute \( a = 4 \) into the original in